In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 82.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=c67216bb745a5631d762b48c3dd2586ea19943988a1d1eff728fce0f0e569cef
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


# BB84 Quantum Key Distribution With Attacker (Eve)

### Eve's intercept-resend attack
Eve intercepts each qubit, picks a random basis (using quantum randomness), measures it, and forwards a **freshly prepared** qubit encoding her measurement result to Bob.  
Because Eve often chooses the wrong basis, she disturbs ~25% of the sifted key bits, which Alice and Bob can detect during error checking.

### Detection
Alice and Bob sacrifice a sample of their sifted key to estimate the error rate.  
An error rate above the threshold (10%) triggers an attack warning and the key exchange is aborted.

In [2]:
%pip install qiskit==1.2.4 qiskit-aer==0.15.1 pylatexenc==2.10 -q

In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import math

# Shared simulator
simulator = AerSimulator()

# Measures n qubits each prepared in |+⟩ = H|0⟩ to get n unbiased random bits.
def quantum_random_bits(n: int) -> list[int]:
    """Return a list of n random bits by measuring qubits in the |+⟩ state.
    Batches requests to stay within the simulator's 29-qubit limit."""
    MAX_BATCH = 29
    bits = []
    remaining = n
    while remaining > 0:
        batch = min(remaining, MAX_BATCH)
        qc = QuantumCircuit(batch, batch)
        qc.h(range(batch))
        qc.measure(range(batch), range(batch))
        job = simulator.run(transpile(qc, simulator), shots=1, memory=True)
        result_str = job.result().get_memory()[0]
        bits.extend(int(b) for b in reversed(result_str))
        remaining -= batch
    return bits[:n]

print("Utility ready. Sample of 8 quantum random bits:", quantum_random_bits(8))

Utility ready. Sample of 8 quantum random bits: [0, 1, 0, 0, 0, 1, 1, 1]


In [4]:
# Alice encoding
# Basis 0 → rectilinear (+);  Basis 1 → diagonal (×)
#   basis 0, bit 0 → |0⟩
#   basis 0, bit 1 → |1⟩
#   basis 1, bit 0 → |+⟩  (H|0⟩)
#   basis 1, bit 1 → |−⟩  (H|1⟩)
N_QUBITS = 100
SAMPLE_FRACTION = 0.2
DETECTION_THRESHOLD = 0.10

def alice_encode(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)
alice_circuits = alice_encode(alice_bits, alice_bases)

print(f"Alice prepared {N_QUBITS} qubits.")
print(f"Alice bits (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]} (0=+, 1=×)")

Alice prepared 100 qubits.
Alice bits (first 20): [0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1]
Alice bases (first 20): [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0] (0=+, 1=×)


In [5]:
# Eve intercept-resend attack
# For each qubit Alice sends, Eve:
#   1. Chooses a random basis (using quantum randomness - H|0⟩ measurement).
#   2. Measures the qubit in that basis.
#   3. Prepares a NEW qubit encoding her measurement result in her chosen basis.
#   4. Forwards that new qubit to Bob.
# When Eve's basis matches Alice's, she gets the right bit and Bob sees no error.
# When they differ, Eve gets a random result → 50% chance Bob's bit is wrong.
# Overall ~25% of sifted bits will be disrupted.
def eve_intercept(circuits: list[QuantumCircuit]) -> tuple[list[QuantumCircuit], list[int], list[int]]:
    n = len(circuits)
    eve_bases = quantum_random_bits(n) # Eve's random basis choices
    eve_bits = []
    tampered = []

    for qc, basis in zip(circuits, eve_bases):
        # Eve measures in her chosen basis
        intercept_qc = qc.copy()
        if basis == 1:
            intercept_qc.h(0) # rotate to diagonal before measuring
        intercept_qc.measure(0, 0)
        job = simulator.run(transpile(intercept_qc, simulator), shots=1, memory=True)
        measured_bit = int(job.result().get_memory()[0])
        eve_bits.append(measured_bit)

        # Eve re-encodes the measured bit and sends to Bob
        resend_qc = QuantumCircuit(1, 1)
        if measured_bit == 1:
            resend_qc.x(0)
        if basis == 1:
            resend_qc.h(0)
        tampered.append(resend_qc)

    return tampered, eve_bases, eve_bits

tampered_circuits, eve_bases, eve_bits = eve_intercept(alice_circuits)

print("Eve has intercepted all qubits and forwarded tampered versions to Bob.")
print(f"Eve bases (first 20): {eve_bases[:20]}")
print(f"Eve bits (first 20): {eve_bits[:20]}")

Eve has intercepted all qubits and forwarded tampered versions to Bob.
Eve bases (first 20): [1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0]
Eve bits (first 20): [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1]


In [6]:
# Bob measurement
def bob_measure(circuits: list[QuantumCircuit], bases: list[int]) -> list[int]:
    results = []
    for qc, basis in zip(circuits, bases):
        meas_qc = qc.copy()
        if basis == 1:
            meas_qc.h(0)
        meas_qc.measure(0, 0)
        job = simulator.run(transpile(meas_qc, simulator), shots=1, memory=True)
        results.append(int(job.result().get_memory()[0]))
    return results

bob_bases = quantum_random_bits(N_QUBITS)
bob_bits  = bob_measure(tampered_circuits, bob_bases)

print(f"Bob measured {N_QUBITS} qubits.")
print(f"Bob bases (first 20): {bob_bases[:20]}")
print(f"Bob bits (first 20): {bob_bits[:20]}")

Bob measured 100 qubits.
Bob bases (first 20): [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1]
Bob bits (first 20): [0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1]


In [7]:
# Sifting - Alice and Bob publicly compare bases
def sift_key(alice_bases, bob_bases, alice_bits, bob_bits):
    alice_sifted, bob_sifted, positions = [], [], []
    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])
            positions.append(i)
    return alice_sifted, bob_sifted, positions

alice_sifted, bob_sifted, matching_positions = sift_key(
    alice_bases, bob_bases, alice_bits, bob_bits
)

print(f"Sifted key length: {len(alice_sifted)} (expected ≈ {N_QUBITS//2})")
print(f"Alice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob sifted (first 20): {bob_sifted[:20]}")

Sifted key length: 49 (expected ≈ 50)
Alice sifted (first 20): [0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0]
Bob sifted (first 20): [0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0]


In [8]:
# Error checking to detect Eve
sample_size = max(1, int(len(alice_sifted) * SAMPLE_FRACTION))
sample_alice = alice_sifted[:sample_size]
sample_bob = bob_sifted[:sample_size]

errors = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / sample_size

# Eve's knowledge
sifted_positions_set = set(matching_positions)
eve_correct = sum(
    1 for i in matching_positions
    if eve_bases[i] == alice_bases[i]
)
eve_knowledge = eve_correct / len(matching_positions) if matching_positions else 0

print("── Error checking ──────────────────────────────────")
print(f"  Sample size          : {sample_size} bits")
print(f"  Errors found         : {errors}")
print(f"  Error rate           : {error_rate:.1%}  (expected ≈25% with full intercept)")
print(f"  Threshold            : {DETECTION_THRESHOLD:.0%}")
print(f"  [Eve's perspective]  : Eve guessed the right basis {eve_knowledge:.1%} of the time (expected ≈50% — she gains ~50% of sifted key but causes ~25% errors)")

if error_rate > DETECTION_THRESHOLD:
    print(" ATTACK DETECTED - error rate too high.")
else:
    print(" No attack detected.")
    final_key = alice_sifted[sample_size:]
    print(f"Final key: {final_key[:20]}...")

── Error checking ──────────────────────────────────
  Sample size          : 9 bits
  Errors found         : 0
  Error rate           : 0.0%  (expected ≈25% with full intercept)
  Threshold            : 10%
  [Eve's perspective]  : Eve guessed the right basis 44.9% of the time (expected ≈50% — she gains ~50% of sifted key but causes ~25% errors)
 No attack detected.
Final key: [0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1]...
